# Stage 2 - Chunking

## Data
1_corpus.json

## Chunking strategies

1. 512 chars with overlap 
  - со сдвигом: chunk_size = 50
  - within text items

2. sentence based 
  - within text items 
  - max_tokens=512

## Install
pip install tiktoken razdel

In [8]:
'''
1. Load Docs
'''
import json

with open("1_corpus.json", encoding="utf-8") as f:
  docs = json.load(f)

print("Разделов:", len(docs))

Разделов: 156


In [9]:
'''
2. Chunking
'''
import tiktoken
from razdel import sentenize  # сегментация русского текста на предложения

ENC = tiktoken.get_encoding("cl100k_base")

def fixed_size_chunks(text, chunk_size=512, overlap=50):
    """Нарезает текст fixed-size-чанкингом: окно из chunk_size токенов со сдвигом chunk_size − overlap."""
    tokens = ENC.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        window = tokens[start:start + chunk_size]
        chunks.append(ENC.decode(window))
        # Сдвигаем окно: на overlap токенов меньше длины окна
        start += chunk_size - overlap
    return chunks


def sentence_chunks(text, max_tokens=512):
    """Нарезает текст sentence-based-чанкингом: набирает целые предложения, пока не упрётся в бюджет max_tokens."""
    sentences = [s.text for s in sentenize(text)]
    chunks = []
    current = []
    current_tokens = 0
    for sent in sentences:
        sent_tokens = len(ENC.encode(sent))
        # Следующее предложение не влезает — закрываем чанк
        if current and current_tokens + sent_tokens > max_tokens:
            chunks.append(" ".join(current))
            current = []
            current_tokens = 0
        current.append(sent)
        current_tokens += sent_tokens
    if current:
        chunks.append(" ".join(current))
    return chunks


def describe(name, chunks):
    """Печатает сводку по списку чанков: количество, средний размер, текст на стыке соседних чанков."""
    sizes = [len(ENC.encode(c)) for c in chunks]
    avg = sum(sizes) / len(sizes)
    end0 = chunks[0][-24:].replace("\n", " ")
    start1 = chunks[1][:24].replace("\n", " ")
    print(f"{name}")
    print(f"  чанков: {len(chunks)}, средний размер: {avg:.0f}")
    print(f"  размеры, токенов: {sizes}")
    print(f"  чанк 1 кончается на: ...{end0!r}")
    print(f"  чанк 2 начинается с: {start1!r}...")
    print()


chunks_fixed = []
chunks_sentence = []
for i in docs:
  fixed = fixed_size_chunks(i['raw'], chunk_size=512, overlap=50)
  chunks_fixed.extend(fixed)
  
  sentence = sentence_chunks(i['raw'], max_tokens=512)
  chunks_sentence.extend(sentence)

# docs as a list
texts = [item["raw"] for item in docs]
texts_str = ", ".join(texts)

print(f"Корпус: {len(ENC.encode(texts_str))} токена\n")
describe("fixed-size (512/50)", chunks_fixed)
describe("sentence-based (512)", chunks_sentence)

Корпус: 51705 токена

fixed-size (512/50)
  чанков: 195, средний размер: 273
  размеры, токенов: [181, 483, 21, 195, 410, 502, 40, 470, 8, 199, 202, 184, 177, 183, 215, 197, 195, 355, 185, 216, 158, 512, 248, 160, 512, 275, 305, 503, 41, 189, 415, 486, 24, 501, 39, 168, 208, 482, 20, 184, 159, 195, 464, 2, 410, 483, 21, 222, 483, 21, 149, 416, 447, 257, 512, 73, 254, 51, 146, 177, 443, 170, 226, 512, 97, 218, 156, 434, 512, 193, 512, 278, 480, 18, 512, 110, 442, 512, 293, 42, 93, 504, 42, 173, 233, 170, 82, 484, 22, 512, 105, 140, 302, 242, 150, 196, 273, 452, 159, 338, 180, 194, 204, 224, 512, 346, 184, 278, 385, 402, 443, 200, 159, 280, 480, 18, 473, 11, 512, 434, 186, 222, 512, 327, 181, 460, 393, 157, 181, 212, 218, 414, 135, 200, 109, 102, 512, 190, 303, 430, 491, 29, 212, 137, 184, 168, 271, 305, 452, 512, 238, 399, 207, 187, 512, 288, 26, 232, 305, 174, 377, 142, 215, 10, 329, 512, 263, 450, 203, 512, 238, 309, 484, 22, 512, 197, 177, 278, 512, 252, 164, 433, 512, 61, 457, 512, 

In [ ]:
import statistics  # Медиана и другая базовая статистика

MIN_TOKENS = 100   # Чанк короче этого порога — подозрительно короткий


def is_truncated(chunk):
    """Возвращает True, если чанк не заканчивается завершающим знаком препинания, — значит, оборван."""
    if chunk.rstrip().endswith((".", "!", "?", "…")):
      return False
    else: 
      # print('>>> Truncated: ', chunk)
      return True


def evaluate(name, chunks):
    """Считает метрики качества разбивки: распределение размеров, число коротких и оборванных чанков."""
    sizes = [len(ENC.encode(c)) for c in chunks]
    short = sum(size < MIN_TOKENS for size in sizes)
    truncated = sum(is_truncated(c) for c in chunks)
    print(f"{name}:")
    print(f"  чанков: {len(chunks)}")
    print(f"  размеры, токенов: {sorted(sizes)}")
    print(f"  min / медиана / max: {min(sizes)} / "
          f"{statistics.median(sizes):.0f} / {max(sizes)}")
    print(f"  короче {MIN_TOKENS} токенов: {short}")
    print(f"  обрезаны посреди предложения: {truncated}")

    for c in chunks:
      if len(ENC.encode(c))  < MIN_TOKENS:
        print('>>> small size: ', c)
    #    if is_truncated(c):
    #     print('>>> truncated: ', c)


evaluate("fixed-size (512/50)", chunks_fixed)
print()
evaluate("sentence-based (512)", chunks_sentence)

fixed-size (512/50):
  чанков: 195
  размеры, токенов: [2, 8, 10, 11, 18, 18, 20, 21, 21, 21, 22, 22, 24, 26, 29, 39, 40, 41, 42, 42, 51, 59, 61, 73, 82, 93, 97, 102, 105, 109, 110, 131, 135, 137, 140, 142, 146, 149, 150, 156, 157, 158, 159, 159, 159, 160, 164, 168, 168, 170, 170, 173, 174, 174, 177, 177, 177, 180, 181, 181, 181, 183, 184, 184, 184, 184, 185, 186, 187, 189, 190, 193, 194, 195, 195, 195, 196, 197, 197, 199, 199, 200, 200, 202, 203, 204, 207, 208, 212, 212, 215, 215, 216, 218, 218, 222, 222, 224, 226, 232, 233, 238, 238, 242, 248, 252, 254, 257, 263, 271, 273, 275, 278, 278, 278, 279, 280, 288, 293, 302, 303, 305, 305, 305, 309, 327, 329, 338, 346, 355, 377, 382, 385, 393, 395, 399, 400, 402, 410, 410, 414, 415, 416, 430, 433, 434, 434, 442, 443, 443, 447, 450, 452, 452, 457, 460, 464, 470, 473, 480, 480, 482, 483, 483, 483, 484, 484, 486, 491, 501, 502, 503, 504, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512

In [4]:
with open("tmp/chunks_list.json", "w", encoding="utf-8") as f:
  json.dump(chunks_sentence, f, ensure_ascii=False, indent=4)

In [10]:
chunks_final = []
id = 0
for i in docs:
  chunks = sentence_chunks(i['raw'], max_tokens=512)

  for j in chunks:
    chunk =   {
      "id":       id,
      "doc_id":   i["doc_id"],
      "title":    i['title'],
      "category": i['category'],
      "format":   i['format'],
      'text':     j,
    }
    chunks_final.append(chunk)
    id += 1
    # print(chunk)
    
with open("2_chunks.json", "w", encoding="utf-8") as f:
  json.dump(chunks_final, f, ensure_ascii=False, indent=4)

# Stage 2 - Report
Sentence chunking is selected.

2_chunks.json